# Notebook 7 — Explainability (signed SHAP)

## What changed from R01

| Change | Reason |
|---|---|
| **Signed** SHAP values retained | Q14: R01 reported mean *absolute* SHAP only, so the submission could not say whether higher family income predicts more dropout or less. Direction is the finding in a dropout model, and R01 discarded it before reporting |
| **Family-level attribution across all features** | Q13: R01 reported a truncated top five plus two named ranks. The full ranking was committed all along, so the truncation had no data cost to reverse |
| Explained on a **CV-held-out fold**, not the frozen test partition | GATE-1(iii). R01 explained on the test partition, which is the right instinct against memorisation but was one of the six notebooks scoring it |
| Both SHAP artefacts **labelled by model** | Q14: `shap_feature_importance.csv` (38 features) and `feature_importance_focal.csv` (41) disagreed by a factor of two on the top feature with no explanation |
| Minority-class attribution uncertainty reported | Q14: attribution rests on ~18 positive instances and R01 stated no uncertainty |
| Imports `losses.py` | R01 redefined the focal loss locally "so this notebook has no dependency on losses.py" — which is precisely how the two copies drifted apart |
| Composite verification is reported *with* interpretation | M17 said the ranking would be "reported in R6 without interpretation". The ablation already showed the composites cost performance, so the verification has an answer and it should be stated |

In [ ]:
# ---- bootstrap: repo-relative imports, no drive.mount, no hard-coded path ----
import sys, os
from pathlib import Path

def _find_repo(start=None):
    p = Path(start or Path.cwd()).resolve()
    for c in [p, *p.parents]:
        if (c / "config.py").exists():
            return c
    return p

REPO = Path(os.environ["DROPOUT_REPO"]) if os.environ.get("DROPOUT_REPO") else _find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# In Colab, clone the repo first, then run:
#     import os; os.environ["DROPOUT_REPO"] = "/content/student-dropout-prediction-ghana"
# Raw pupil-level data is NOT in the repo (ethics); place it under data-raw/
# locally. Nothing below calls drive.mount().

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from config import *
from pipeline import (preprocess_inside_fold, frozen_split, cv_splits,
                      fit_arm, ARMS, ARM_LABELS, HEADLINE, OLD_HEADLINE,
                      raw_feature_cols)

banner("NOTEBOOK 7 — SIGNED SHAP")
OUT = run_dir("notebook07_shap")
capture_environment(OUT)

try:
    import shap
except ImportError:
    raise ImportError("pip install shap")

df = pd.read_csv(CLEANED_CSV)
train_pool, test_holdout = frozen_split(df)

# Explain on a CV-held-out fold of the training pool. The frozen test
# partition stays untouched until Notebook 8.
tr, vl = cv_splits(train_pool, SPLIT_SEED, n_repeats=1)[0]
X_tr, y_tr, X_vl, y_vl, meta = preprocess_inside_fold(
    train_pool.iloc[tr], train_pool.iloc[vl])
N_POS_EXPLAINED = int((y_vl == 1).sum())
print(f"explaining on {len(y_vl)} held-out rows, {N_POS_EXPLAINED} of them dropout")
print(f"{meta['n_features']} features")
print("\nThe test partition is NOT used here.")

In [ ]:
# ---- signed SHAP per arm, each artefact labelled ----------------------
def shap_for_arm(arm_name):
    model, predict, cols = fit_arm(arm_name, X_tr, y_tr, SPLIT_SEED)
    sv = shap.TreeExplainer(model).shap_values(X_vl[cols])
    if isinstance(sv, list):
        sv = sv[1]
    sv = np.asarray(sv)
    if sv.ndim == 3:
        sv = sv[:, :, 1]

    pos = (y_vl.to_numpy() == 1)
    t = pd.DataFrame({
        "feature": cols,
        "mean_abs_shap": np.abs(sv).mean(axis=0),
        "mean_signed_shap": sv.mean(axis=0),
        "sd_shap": sv.std(axis=0),
        "mean_abs_shap_positives": (np.abs(sv[pos]).mean(axis=0)
                                    if pos.sum() else np.nan),
        "mean_signed_shap_positives": (sv[pos].mean(axis=0)
                                       if pos.sum() else np.nan),
        "se_signed_positives": (sv[pos].std(axis=0, ddof=1) / np.sqrt(pos.sum())
                                if pos.sum() > 1 else np.nan),
    })
    t["direction"] = np.where(t["mean_signed_shap"] > 0, "raises dropout risk",
                       np.where(t["mean_signed_shap"] < 0, "lowers dropout risk",
                                "flat"))
    t["family"] = t["feature"].map(family_of)
    t["is_composite"] = t["feature"].isin(COMPOSITES)
    # MODEL LABEL — the fix for the two disagreeing R01 artefacts
    t["model_arm"] = arm_name
    t["model_label"] = ARM_LABELS[arm_name]
    t["n_features_in_model"] = len(cols)
    t["n_positive_instances_explained"] = N_POS_EXPLAINED
    t = t.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    t["rank"] = t.index + 1
    return t, sv, cols

tables, raw_sv = {}, {}
for arm in [HEADLINE[1], HEADLINE[0], OLD_HEADLINE[0]]:
    t, sv, cols = shap_for_arm(arm)
    tables[arm] = t
    raw_sv[arm] = (sv, cols)
    fn = f"shap_signed__{arm}.csv"
    t.to_csv(OUT / fn, index=False)
    print(f"\n=== {arm} — {ARM_LABELS[arm]} ===")
    print(f"    {len(cols)} features, file: {fn}")
    print(t.head(12)[["rank", "feature", "family", "mean_abs_shap",
                      "mean_signed_shap", "direction"]].round(4).to_string(index=False))

In [ ]:
# ---- reconcile the two R01 artefacts (Q14) ---------------------------
a, b = HEADLINE[1], OLD_HEADLINE[0]
rec = (tables[a][["feature", "rank", "mean_abs_shap"]]
       .merge(tables[b][["feature", "rank", "mean_abs_shap"]],
              on="feature", how="outer", suffixes=(f"__{a}", f"__{b}")))
rec["abs_shap_ratio"] = (rec[f"mean_abs_shap__{a}"] / rec[f"mean_abs_shap__{b}"])
rec = rec.sort_values(f"mean_abs_shap__{a}", ascending=False)
rec.to_csv(OUT / "shap_artefact_reconciliation.csv", index=False)
print("SHAP ARTEFACT RECONCILIATION\n")
print(rec.head(12).round(4).to_string(index=False))
print(f"""
R01 committed two SHAP files that disagreed by a factor of two on the top
feature, with nothing in the manuscript saying which model produced which.
They are coherent as the baseline ({b}, {tables[b]['n_features_in_model'].iloc[0]} features) and the
proposed model ({a}, {tables[a]['n_features_in_model'].iloc[0]} features). Every file this notebook writes
carries model_arm and model_label columns, and the filename names the arm.
Attribution magnitudes are NOT comparable across models with different
feature counts — say that wherever both are cited.""")

In [ ]:
# ---- family-level attribution (Q13) ---------------------------------
arm = HEADLINE[1]
t = tables[arm]
fam = (t.groupby("family")
       .agg(total_abs_shap=("mean_abs_shap", "sum"),
            mean_abs_shap=("mean_abs_shap", "mean"),
            mean_signed=("mean_signed_shap", "mean"),
            n_features=("feature", "size"),
            top_feature=("feature", "first"))
       .sort_values("total_abs_shap", ascending=False).reset_index())
fam["share_pct"] = (100 * fam["total_abs_shap"] / fam["total_abs_shap"].sum()).round(1)
fam["model_arm"] = arm
fam.to_csv(OUT / f"shap_family_attribution__{arm}.csv", index=False)

print("FAMILY-LEVEL ATTRIBUTION — replaces the top-five list in R6")
print(f"(all {len(t)} features, {arm})\n")
print(fam.round(4).to_string(index=False))

unclass = t[t["family"] == "other_unclassified"]
if len(unclass):
    print(f"\n{len(unclass)} features unclassified — add keywords to "
          f"config.FEATURE_FAMILIES:")
    print("   ", ", ".join(unclass["feature"].head(15)))

if SCHOOL_COL in set(t["feature"]):
    r = t[t["feature"] == SCHOOL_COL].iloc[0]
    print(f"\n*** {SCHOOL_COL} ranks {int(r['rank'])} of {len(t)} "
          f"(mean |SHAP| {r['mean_abs_shap']:.4f}) ***")
    print("An administrative identifier is not a construct — it is a cluster "
          "label, and its presence means the model has learned which school a "
          "record came from. Report it SEPARATELY from the construct families "
          "and say what it means. Setting config.SCHOOL_HANDLING='drop' "
          "removes it; that is decision D1.")
else:
    print(f"\n{SCHOOL_COL} excluded from the feature matrix "
          f"(SCHOOL_HANDLING={SCHOOL_HANDLING!r}), so no school-administrative "
          "family appears. State this in M9 and R6.")

plt.figure(figsize=(8, 4))
o = fam.sort_values("total_abs_shap")
plt.barh(o["family"], o["total_abs_shap"], color="steelblue")
plt.xlabel("summed mean |SHAP|"); plt.title(f"Family attribution — {arm}")
plt.tight_layout(); plt.savefig(OUT / "figures/family_attribution.png", dpi=200)
plt.close()

In [ ]:
# ---- DIRECTION, and the theory test (Q13, Q14) -----------------------
print("SIGNED DIRECTION OF THE TOP 15 FEATURES")
print("This is the table R01 could not produce, and it is what a domain")
print("examiner asks about first: 'which way does it point?'\n")
top = t.head(15)[["rank", "feature", "family", "mean_abs_shap",
                  "mean_signed_shap", "direction", "se_signed_positives"]]
print(top.round(4).to_string(index=False))

drift = top.copy()
drift["expected_direction"] = ""      # fill from your adopted theory
drift["matches_expectation"] = ""     # yes / no
drift["diagnosis"] = ""               # operationalisation / data / theory misfit
drift.to_csv(OUT / "construct_drift_worksheet.csv", index=False)
print("\nconstruct_drift_worksheet.csv written — fill the three blank columns. "
      "Q14 asks you to classify each mismatch as an operationalisation "
      "failure, a data failure, or a theory misfit.")

# The candidate drift the examination identified, now checkable
sv_col = "socioeconomic_vulnerability_score"
inc_col = SOCIOECONOMIC_COLS["family_income"]
for c in (sv_col, inc_col):
    if c in set(t["feature"]):
        r = t[t["feature"] == c].iloc[0]
        print(f"\n{c}: rank {int(r['rank'])}/{len(t)}, "
              f"mean |SHAP| {r['mean_abs_shap']:.4f}, {r['direction']}")
if sv_col in set(t["feature"]) and inc_col in set(t["feature"]):
    rs = int(t[t['feature']==sv_col]['rank'].iloc[0])
    ri = int(t[t['feature']==inc_col]['rank'].iloc[0])
    print(f"""
R01 pattern: the engineered composite ranked 24th of 41 (0.017) while its raw
ingredient family_income_level ranked 4th (0.225). Engineered construct dead,
raw ingredient alive — the signature of a broken operationalisation, not of a
theory being wrong about the world. The cause was the alphabetical encoding
(see Notebook 3). Current ranks: composite {rs}, raw ingredient {ri}.
If the composite has climbed, the fault was the encoding. If it is still
inert, the composite's construction is wrong and that is worth reporting.""")

In [ ]:
# ---- beeswarm and composite verification (M17) -----------------------
sv, cols = raw_sv[arm]
plt.figure()
shap.summary_plot(sv, X_vl[cols], max_display=20, show=False)
plt.title(f"Signed SHAP — {ARM_LABELS[arm]} (raw log-odds)")
plt.tight_layout()
plt.savefig(OUT / "figures/shap_beeswarm.png", dpi=300, bbox_inches="tight")
plt.close()

plt.figure()
shap.summary_plot(sv, X_vl[cols], plot_type="bar", max_display=20, show=False)
plt.title(f"mean |SHAP| — {ARM_LABELS[arm]}")
plt.tight_layout()
plt.savefig(OUT / "figures/shap_bar.png", dpi=300, bbox_inches="tight")
plt.close()
print("figures ->", OUT / "figures")

print("\nCOMPOSITE VERIFICATION (M17's stated purpose)")
comp = t[t["is_composite"]][["rank", "feature", "mean_abs_shap",
                             "mean_signed_shap", "direction"]]
print(comp.round(4).to_string(index=False))
n_raw = len(raw_feature_cols(X_vl))
print(f"\n{len(comp)} composites among {len(t)} features "
      f"({n_raw} raw). Composite ranks: {sorted(comp['rank'].tolist())}")
print("""
M17 said this ranking would be "reported in R6 without interpretation". It
has an interpretation and withholding it does not help: the ablation already
shows the composites reduced AUC-PR. A composite that ranks high in SHAP
while the ablation says it costs performance means the model uses it and is
worse for doing so -- which is redundancy with the raw ingredients, not
signal. Say that, and cite both tables.""")

write_manifest(OUT, {"notebook": "07_shap", "test_set_scored": False,
                     "explained_on": "CV-held-out fold of the training pool",
                     "n_instances_explained": int(len(y_vl)),
                     "n_positive_explained": N_POS_EXPLAINED,
                     "arms_explained": list(tables),
                     "signed_values_retained": True})